<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/LongCat-Video-Avatar-1.5_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/LongCat-Video-Avatar-1.5_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# LongCat-Video-Avatar 1.5 — Audio-Driven Avatar Video Studio (Colab Edition)

End-to-end Google Colab notebook for **[meituan-longcat/LongCat-Video-Avatar-1.5](https://huggingface.co/meituan-longcat/LongCat-Video-Avatar-1.5)** (14B parameter 3D Diffusion Transformer, 25 FPS native output) incorporating the INT8 + DMD2 8-step distillation + DBCache optimizations from **[victor/LongCat-Video-Avatar-1.5](https://huggingface.co/spaces/victor/LongCat-Video-Avatar-1.5)** plus Colab-specific host-RAM and GPU-VRAM engineering.

### Key Capabilities
- **Single-Avatar Image + Audio to Video (`AI2V`)**: Animate any portrait or character image with expressive lip-sync, head motion, and hand gestures driven by speech or singing audio + text prompt.
- **Seamless Long-Video Continuation (`AVC`)**: Extend avatar videos across arbitrary audio lengths using 13-frame latent overlap conditioning (`num_cond_frames=13`), KV-caching (`offload_kv_cache`), and Reference Skip Attention (`ref_img_index=10`, `mask_frame_range=3`) to prevent identity drift or repetitive gestures.
- **Audio + Text to Video (`AT2V`)**: Synthesize a speaking character and environment from scratch using only an audio clip and text prompt (no reference image required).
- **Dual-Speaker Conversation (`Multi-AI2V`)**: Drive two characters in the same frame from two separate audio tracks (`audio_type="add"` for turn-taking dialogue or `"para"` for simultaneous duet/chorus) using vectorized spatial bounding-box attention masks.

### Pipeline Architecture & Colab Optimizations
```
Input Speech (.wav/.mp3) ──► Kim_Vocal_2 (ONNX) ──► Whisper-Large-v3 (25fps Audio Tokens) ─┐
Reference Image (optional) ──► Wan 2.1 3D VAE Encode (16-ch Latents) ──────────────────────┼──► LongCat-Avatar-1.5 3D DiT (INT8 + DMD2 8-Step LoRA + DBCache)
Scene & Action Prompt ───────► UMT5-XXL Text Encoder (Disk & RAM Cached) ──────────────────┘             │
                                                                                                         ▼
Final MP4 (25 FPS) ◄── FFmpeg Stream Copy Mux ◄── Wan 2.1 3D VAE Decode (Stage-Offloaded) ◄──────────────┘
```

| GPU Tier | VRAM | Host RAM | Recommended `VRAM_MODE` | Peak VRAM (`480p`, 93f, 8 steps) |
| :--- | :--- | :--- | :--- | :--- |
| **Colab L4** | 24 GB | 53 GB | `stage-offload (L4 24GB)` *(auto)* | **~18.2 GB** (100% native GPU DiT speed) |
| **Colab A100** | 40 GB / 80 GB | 83 GB+ | `full-gpu (A100 40GB/80GB)` *(auto)* | **~30.5 GB** (all encoders resident) |

> **Why standard `load_quantized_dit` crashes Colab L4 (and how Step 3 fixes it):**
> Upstream `load_quantized_dit()` instantiates the 48-block `LongCatVideoAvatarTransformer3DModel` in FP32 on CPU (~54 GB RAM) before replacing `nn.Linear` with `QuantizedLinear` and loading all 4 `.safetensors` shards into a single dict (~69 GB peak host RAM — exceeding L4's 53 GB limit). **Step 3** builds the model skeleton on PyTorch's zero-memory `meta` device (`0 MB`) and streams each shard directly via `load_state_dict(..., assign=True)`, keeping peak host RAM under **~19 GB**.

In [ ]:
#@title STEP 1 — Install Environment & Clone LongCat-Video-Avatar 1.5
CONNECT_GOOGLE_DRIVE = True #@param {type:"boolean"}
INSTALL_ONNXRUNTIME_GPU = True #@param {type:"boolean"}

import os
import sys
import shutil
import subprocess
import types
import urllib.request
from pathlib import Path

if CONNECT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        DRIVE_ROOT = Path("/content/drive/MyDrive")
    except Exception as exc:
        print(f"[WARN] Google Drive mount unavailable ({exc}); using /content/local_drive")
        DRIVE_ROOT = Path("/content/local_drive")
else:
    DRIVE_ROOT = Path("/content/local_drive")

CACHE_ROOT = DRIVE_ROOT / "AEI_3D_Cache" / "LongCat-Video-Avatar-1.5"
OUT_DIR = DRIVE_ROOT / "AEI_3D_Out" / "LongCat-Video-Avatar-1.5"
REPO_DIR = Path("/content/LongCat-Video-Avatar-1.5")

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_ROOT / "hf_home")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"Cache root : {CACHE_ROOT}")
print(f"Output dir : {OUT_DIR}")
print(f"Repo dir   : {REPO_DIR}")

# 1. Clone victor/LongCat-Video-Avatar-1.5 Space repository (contains longcat_video/ + DBCache + INT8 loader)
if not (REPO_DIR / "longcat_video").exists():
    print("\n[1/4] Cloning https://huggingface.co/spaces/victor/LongCat-Video-Avatar-1.5 ...")
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://huggingface.co/spaces/victor/LongCat-Video-Avatar-1.5", str(REPO_DIR)],
        check=True,
    )
else:
    print("\n[1/4] Repository already cloned at", REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 2. Ensure sample assets (both Single-Avatar and Multi-Person examples) are materialized (not LFS pointer stubs)
print("[2/4] Verifying example portrait & audio assets ...")
SPACE_RAW = "https://huggingface.co/spaces/victor/LongCat-Video-Avatar-1.5/resolve/main"
GH_RAW = "https://raw.githubusercontent.com/meituan-longcat/LongCat-Video/main"

ASSET_MANIFEST = [
    (f"{SPACE_RAW}/assets/avatar/single/character.png", REPO_DIR / "assets/avatar/single/character.png"),
    (f"{SPACE_RAW}/assets/avatar/single/hello.wav", REPO_DIR / "assets/avatar/single/hello.wav"),
    (f"{SPACE_RAW}/assets/avatar/examples/photoreal_person.png", REPO_DIR / "assets/avatar/examples/photoreal_person.png"),
    (f"{SPACE_RAW}/assets/avatar/examples/orc_warrior.png", REPO_DIR / "assets/avatar/examples/orc_warrior.png"),
    (f"{SPACE_RAW}/assets/avatar/examples/audio/photoreal_person_voice.wav", REPO_DIR / "assets/avatar/examples/audio/photoreal_person_voice.wav"),
    (f"{SPACE_RAW}/assets/avatar/examples/audio/orc_warrior_voice.wav", REPO_DIR / "assets/avatar/examples/audio/orc_warrior_voice.wav"),
    (f"{SPACE_RAW}/assets/avatar/examples/audio/character_voice.wav", REPO_DIR / "assets/avatar/examples/audio/character_voice.wav"),
    (f"{GH_RAW}/assets/avatar/multi/sing.png", REPO_DIR / "assets/avatar/multi/sing.png"),
    (f"{GH_RAW}/assets/avatar/multi/sing_man.WAV", REPO_DIR / "assets/avatar/multi/sing_man.WAV"),
    (f"{GH_RAW}/assets/avatar/multi/sing_woman.WAV", REPO_DIR / "assets/avatar/multi/sing_woman.WAV"),
    (f"{GH_RAW}/assets/avatar/multi/introduce.png", REPO_DIR / "assets/avatar/multi/introduce.png"),
    (f"{GH_RAW}/assets/avatar/multi/introduce_woman.mp3", REPO_DIR / "assets/avatar/multi/introduce_woman.mp3"),
    (f"{GH_RAW}/assets/avatar/multi/introduce_man.mp3", REPO_DIR / "assets/avatar/multi/introduce_man.mp3"),
    (f"{GH_RAW}/assets/avatar/multi_example_1.json", REPO_DIR / "assets/avatar/multi_example_1.json"),
    (f"{GH_RAW}/assets/avatar/multi_example_2.json", REPO_DIR / "assets/avatar/multi_example_2.json"),
]

for url, dest in ASSET_MANIFEST:
    dest.parent.mkdir(parents=True, exist_ok=True)
    needs_dl = not dest.exists() or dest.stat().st_size < 256
    if not needs_dl and dest.suffix.lower() in (".png", ".wav", ".mp3"):
        with open(dest, "rb") as fh:
            header = fh.read(64)
        if b"git-lfs.github.com" in header:
            needs_dl = True
    if needs_dl:
        try:
            urllib.request.urlretrieve(url, str(dest))
        except Exception as exc:
            print(f"  [WARN] Could not fetch example asset {dest.name}: {exc}")

# 3. Install Python dependencies (Colab Python 3.12 safe — preserves pre-installed PyTorch/CUDA)
print("[3/4] Installing Python dependencies ...")
CORE_PKGS = [
    "huggingface-hub>=0.27.0",
    "transformers>=4.49.0",
    "diffusers>=0.32.2",
    "accelerate>=1.0.0",
    "safetensors>=0.4.0",
    "einops>=0.8.0",
    "loguru",
    "ftfy",
    "regex",
    "imageio",
    "imageio-ffmpeg",
    "opencv-python-headless",
    "soundfile",
    "librosa",
    "pyloudnorm",
    "audio-separator>=0.30.2",
    "onnx>=1.16.0",
    "gradio>=5.0.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + CORE_PKGS, check=True)

if INSTALL_ONNXRUNTIME_GPU:
    ort_res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu"])
    if ort_res.returncode != 0:
        print("  [INFO] onnxruntime-gpu wheel unavailable; falling back to CPU onnxruntime ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime>=1.18.0"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime>=1.18.0"], check=True)

# 4. Register ZeroGPU 'spaces' stub so any @spaces.GPU decorator is a harmless pass-through
if "spaces" not in sys.modules:
    spaces_stub = types.ModuleType("spaces")
    def _gpu_passthrough(fn=None, **_kwargs):
        if fn is None:
            return lambda f: f
        return fn
    spaces_stub.GPU = _gpu_passthrough
    sys.modules["spaces"] = spaces_stub

import torch
print("\n[4/4] Environment ready!")
print(f"  Python  : {sys.version.split()[0]}")
print(f"  PyTorch : {torch.__version__} (CUDA available: {torch.cuda.is_available()})")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"  GPU     : {gpu_name} ({vram_gb:.1f} GB VRAM)")

In [ ]:
#@title STEP 2 — Download & Cache Model Weights to Google Drive (~42.3 GB Filtered)
DOWNLOAD_BASE_BF16_DIT = False #@param {type:"boolean"}
HF_TOKEN_BELOW = "" #@param {type:"string"}

import json
import os
from pathlib import Path
from huggingface_hub import snapshot_download

if "CACHE_ROOT" not in globals():
    CACHE_ROOT = Path("/content/drive/MyDrive/AEI_3D_Cache/LongCat-Video-Avatar-1.5")
    if not Path("/content/drive").exists():
        CACHE_ROOT = Path("/content/local_drive/AEI_3D_Cache/LongCat-Video-Avatar-1.5")

BASE_DIR = CACHE_ROOT / "LongCat-Video"
AVATAR_DIR = CACHE_ROOT / "LongCat-Video-Avatar-1.5"
BASE_DIR.mkdir(parents=True, exist_ok=True)
AVATAR_DIR.mkdir(parents=True, exist_ok=True)

hf_token = HF_TOKEN_BELOW.strip() or os.environ.get("HF_TOKEN") or None

print("=== [1/2] Downloading shared components from meituan-longcat/LongCat-Video ===")
print("  (tokenizer, UMT5-XXL text_encoder, Wan 2.1 3D VAE -> ~22.2 GB)")
snapshot_download(
    repo_id="meituan-longcat/LongCat-Video",
    local_dir=str(BASE_DIR),
    token=hf_token,
    allow_patterns=[
        "tokenizer/*",
        "text_encoder/*.safetensors",
        "text_encoder/*.json",
        "vae/*.safetensors",
        "vae/*.json",
    ],
    ignore_patterns=[
        "text_encoder/*.fp32*",
        "text_encoder/*.bin",
        "text_encoder/flax_model*",
        "text_encoder/tf_model*",
        "vae/flax_model*",
        "vae/tf_model*",
    ],
)

avatar_allow_patterns = [
    "base_model_int8/*",
    "lora/*",
    "scheduler/*",
    "vocal_separator/*",
    "whisper-large-v3/model.safetensors",
    "whisper-large-v3/*.json",
    "whisper-large-v3/*.txt",
]
if DOWNLOAD_BASE_BF16_DIT:
    avatar_allow_patterns.append("base_model/*")

print("\n=== [2/2] Downloading Avatar 1.5 weights from meituan-longcat/LongCat-Video-Avatar-1.5 ===")
print(
    "  (base_model_int8, DMD2 8-step LoRA, scheduler, Kim_Vocal_2 ONNX, whisper-large-v3 -> ~20.1 GB"
    + (" + 29.5 GB base_model BF16)" if DOWNLOAD_BASE_BF16_DIT else ")")
)
snapshot_download(
    repo_id="meituan-longcat/LongCat-Video-Avatar-1.5",
    local_dir=str(AVATAR_DIR),
    token=hf_token,
    allow_patterns=avatar_allow_patterns,
    ignore_patterns=[
        "whisper-large-v3/model.fp32*",
        "whisper-large-v3/flax_model*",
        "whisper-large-v3/tf_model*",
        "whisper-large-v3/pytorch_model*",
    ],
)

# Patch DiT config.json files to route attention through our PyTorch 2.x SDPA shim (enable_xformers=True)
for sub in ("base_model_int8", "base_model"):
    cfg_path = AVATAR_DIR / sub / "config.json"
    if cfg_path.exists():
        with open(cfg_path, "r", encoding="utf-8") as fh:
            cfg = json.load(fh)
        cfg["enable_flashattn2"] = False
        cfg["enable_flashattn3"] = False
        cfg["enable_bsa"] = False
        cfg["enable_xformers"] = True
        with open(cfg_path, "w", encoding="utf-8") as fh:
            json.dump(cfg, fh, indent=2)
        print(f"  Patched attention flags in {cfg_path.relative_to(CACHE_ROOT)} -> SDPA shim enabled")

# Pre-flight verification summary
print("\n=== Pre-Flight Cache Verification ===")
def _dir_stats(folder: Path):
    if not folder.exists():
        return 0, 0.0
    files = [p for p in folder.rglob("*") if p.is_file() and not p.name.startswith(".")]
    total_gb = sum(p.stat().st_size for p in files) / (1024 ** 3)
    return len(files), total_gb

check_targets = [
    ("Tokenizer (UMT5)", BASE_DIR / "tokenizer"),
    ("Text Encoder (UMT5-XXL)", BASE_DIR / "text_encoder"),
    ("Video VAE (AutoencoderKLWan)", BASE_DIR / "vae"),
    ("Avatar DiT INT8 (base_model_int8)", AVATAR_DIR / "base_model_int8"),
    ("DMD2 8-Step LoRA (lora)", AVATAR_DIR / "lora"),
    ("FlowMatch Scheduler", AVATAR_DIR / "scheduler"),
    ("Vocal Separator (Kim_Vocal_2)", AVATAR_DIR / "vocal_separator"),
    ("Audio Encoder (whisper-large-v3)", AVATAR_DIR / "whisper-large-v3"),
]
if DOWNLOAD_BASE_BF16_DIT:
    check_targets.append(("Avatar DiT BF16 (base_model)", AVATAR_DIR / "base_model"))

total_cache_gb = 0.0
for label, path in check_targets:
    n_files, size_gb = _dir_stats(path)
    total_cache_gb += size_gb
    status = "OK" if n_files > 0 else "MISSING"
    print(f"  [{status:7s}] {label:34s} : {n_files:2d} file(s), {size_gb:6.2f} GB")
print(f"  Total cached model footprint: {total_cache_gb:.2f} GB at {CACHE_ROOT}")

In [ ]:
#@title STEP 3 — Load Pipeline, Zero-Spike INT8 Loader & Stage-Wise VRAM Manager
VRAM_MODE = "auto" #@param ["auto", "stage-offload (L4 24GB)", "full-gpu (A100 40GB/80GB)"]
USE_INT8 = True #@param {type:"boolean"}
USE_DISTILL_LORA = True #@param {type:"boolean"}

import gc
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import tempfile
import time
import types
from collections import OrderedDict
from pathlib import Path

import imageio
import librosa
import numpy as np
import PIL.Image
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers.utils import load_image
from safetensors.torch import load_file
from transformers import AutoTokenizer, UMT5EncoderModel

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

REPO_DIR = Path("/content/LongCat-Video-Avatar-1.5")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if "CACHE_ROOT" not in globals():
    CACHE_ROOT = Path("/content/drive/MyDrive/AEI_3D_Cache/LongCat-Video-Avatar-1.5")
    if not Path("/content/drive").exists():
        CACHE_ROOT = Path("/content/local_drive/AEI_3D_Cache/LongCat-Video-Avatar-1.5")
if "OUT_DIR" not in globals():
    OUT_DIR = Path("/content/drive/MyDrive/AEI_3D_Out/LongCat-Video-Avatar-1.5")
    if not Path("/content/drive").exists():
        OUT_DIR = Path("/content/local_drive/AEI_3D_Out/LongCat-Video-Avatar-1.5")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_DIR = CACHE_ROOT / "LongCat-Video"
AVATAR_DIR = CACHE_ROOT / "LongCat-Video-Avatar-1.5"


def torch_gc():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# ---------------------------------------------------------------------------
# 1. Pure PyTorch 2.x SDPA Shim for xformers.ops.memory_efficient_attention
# ---------------------------------------------------------------------------
def _install_sdpa_shim():
    class _BDShim:
        def __init__(self, q_seqlen, kv_seqlen):
            self.q_seqlen = list(q_seqlen)
            self.kv_seqlen = list(kv_seqlen)

        @classmethod
        def from_seqlens(cls, q_seqlen, kv_seqlen):
            return cls(q_seqlen, kv_seqlen)

    def _meff_sdpa(q, k, v, attn_bias=None, op=None, scale=None, **_):
        # Input layout from LongCat: [B, S, H, D]
        if attn_bias is None:
            q_t = q.transpose(1, 2).contiguous()
            k_t = k.transpose(1, 2).contiguous()
            v_t = v.transpose(1, 2).contiguous()
            return F.scaled_dot_product_attention(q_t, k_t, v_t, scale=scale).transpose(1, 2)
        if isinstance(attn_bias, _BDShim):
            outs, q_off, k_off = [], 0, 0
            for q_len, k_len in zip(attn_bias.q_seqlen, attn_bias.kv_seqlen):
                q_b = q[:, q_off : q_off + q_len].transpose(1, 2).contiguous()
                k_b = k[:, k_off : k_off + k_len].transpose(1, 2).contiguous()
                v_b = v[:, k_off : k_off + k_len].transpose(1, 2).contiguous()
                outs.append(F.scaled_dot_product_attention(q_b, k_b, v_b, scale=scale).transpose(1, 2))
                q_off += q_len
                k_off += k_len
            return torch.cat(outs, dim=1)
        q_t = q.transpose(1, 2).contiguous()
        k_t = k.transpose(1, 2).contiguous()
        v_t = v.transpose(1, 2).contiguous()
        return F.scaled_dot_product_attention(q_t, k_t, v_t, attn_bias=attn_bias, scale=scale).transpose(1, 2)

    xf_mod = sys.modules.get("xformers") or types.ModuleType("xformers")
    ops_mod = sys.modules.get("xformers.ops") or types.ModuleType("xformers.ops")
    fmha_mod = sys.modules.get("xformers.ops.fmha") or types.ModuleType("xformers.ops.fmha")
    bias_mod = sys.modules.get("xformers.ops.fmha.attn_bias") or types.ModuleType("xformers.ops.fmha.attn_bias")

    bias_mod.BlockDiagonalMask = _BDShim
    fmha_mod.attn_bias = bias_mod
    ops_mod.fmha = fmha_mod
    ops_mod.memory_efficient_attention = _meff_sdpa
    xf_mod.ops = ops_mod

    sys.modules["xformers"] = xf_mod
    sys.modules["xformers.ops"] = ops_mod
    sys.modules["xformers.ops.fmha"] = fmha_mod
    sys.modules["xformers.ops.fmha.attn_bias"] = bias_mod


_install_sdpa_shim()
print("[OK] Installed PyTorch 2.x SDPA shim for xformers attention")

# ---------------------------------------------------------------------------
# 2. Zero-Spike Meta-Device INT8 DiT Loader & Vectorized Multi-Speaker Patch
# ---------------------------------------------------------------------------
import longcat_video.modules.quantization as lc_quant
import longcat_video.audio_process.torch_utils as lc_torch_utils
from longcat_video.modules.quantization import QuantizedLinear, DEFAULT_SKIP_PATTERNS
from longcat_video.modules.avatar.longcat_video_dit_avatar import LongCatVideoAvatarTransformer3DModel
from longcat_video.modules.scheduling_flow_match_euler_discrete import FlowMatchEulerDiscreteScheduler
from longcat_video.modules.autoencoder_kl_wan import AutoencoderKLWan
from longcat_video.pipeline_longcat_video_avatar import LongCatVideoAvatarPipeline
from longcat_video.audio_process import get_audio_encoder, get_audio_feature_extractor
from longcat_video.audio_process.torch_utils import save_video_ffmpeg
from audio_separator.separator import Separator


def _fast_calculate_x_ref_attn_map(noise_q, ref_k, ref_target_masks, attn_bias=None):
    """Vectorized, low-VRAM replacement for calculate_x_ref_attn_map in dual-speaker mode.
    Eliminates @torch.compile graph breaks, 3x tensor clones, and 3,072 cuda.empty_cache() stalls.
    """
    ref_k = ref_k.to(dtype=noise_q.dtype, device=noise_q.device)
    scale = 1.0 / (noise_q.shape[-1] ** 0.5)
    nq = (noise_q * scale).transpose(1, 2)
    rk = ref_k.transpose(1, 2)
    attn = nq @ rk.transpose(-2, -1)
    if attn_bias is not None:
        attn = attn + attn_bias
    attn_mean_h = attn.softmax(dim=-1).to(noise_q.dtype).mean(dim=1)  # [B, S_noise, S_ref]
    del attn
    masks = ref_target_masks.to(dtype=noise_q.dtype, device=noise_q.device)
    norm_masks = masks / masks.sum(dim=-1, keepdim=True).clamp_min(1e-6)  # [M, S_ref]
    return torch.matmul(attn_mean_h[0], norm_masks.t()).transpose(0, 1).contiguous()


lc_torch_utils.calculate_x_ref_attn_map = _fast_calculate_x_ref_attn_map


def load_quantized_dit_fast(checkpoint_dir: str, subfolder: str = "base_model_int8", **kwargs):
    """Instantiate LongCatVideoAvatarTransformer3DModel on meta device (0 MB RAM) and stream INT8 shards."""
    quantized_dir = os.path.join(checkpoint_dir, subfolder)
    config_path = os.path.join(quantized_dir, "config.json")
    with open(config_path, "r", encoding="utf-8") as fh:
        config = json.load(fh)

    for k in ("_class_name", "architectures", "_diffusers_version", "model_max_length"):
        config.pop(k, None)
    config["enable_flashattn2"] = False
    config["enable_flashattn3"] = False
    config["enable_bsa"] = False
    config["enable_xformers"] = True
    config.update(kwargs)

    print("  [INT8 Loader] Building 48-block DiT skeleton on meta device (0 MB host RAM) ...")
    with torch.device("meta"):
        model = LongCatVideoAvatarTransformer3DModel(**config)
        modules_to_replace = {}
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                if not any(pat in name for pat in DEFAULT_SKIP_PATTERNS):
                    modules_to_replace[name] = QuantizedLinear(
                        module.in_features, module.out_features, bias=module.bias is not None
                    )
        for name, ql in modules_to_replace.items():
            parts = name.split(".")
            parent = model
            for part in parts[:-1]:
                parent = getattr(parent, part)
            setattr(parent, parts[-1], ql)

    index_path = os.path.join(quantized_dir, "quantized_model.safetensors.index.json")
    if os.path.exists(index_path):
        with open(index_path, "r", encoding="utf-8") as fh:
            index = json.load(fh)
        shard_files = sorted(set(index["weight_map"].values()))
    else:
        shard_files = sorted(
            f for f in os.listdir(quantized_dir) if f.endswith(".safetensors") and "index" not in f
        )

    for idx, shard_file in enumerate(shard_files, 1):
        shard_path = os.path.join(quantized_dir, shard_file)
        print(f"  [INT8 Loader] Streaming shard {idx}/{len(shard_files)}: {shard_file} ...")
        shard_dict = load_file(shard_path, device="cpu")
        for k, tensor in list(shard_dict.items()):
            if tensor.dtype == torch.float32 and not k.endswith(".weight_scale"):
                shard_dict[k] = tensor.to(torch.bfloat16)
        model.load_state_dict(shard_dict, strict=False, assign=True)
        del shard_dict
        gc.collect()

    for name, param in model.named_parameters():
        if param.device.type == "meta":
            raise RuntimeError(f"Parameter {name} remained on meta device after shard loading")
    for name, buf in model.named_buffers():
        if buf.device.type == "meta":
            raise RuntimeError(f"Buffer {name} remained on meta device after shard loading")

    model.eval()
    return model


lc_quant.load_quantized_dit = load_quantized_dit_fast

# ---------------------------------------------------------------------------
# 3. Determine VRAM Mode & Load Models
# ---------------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
total_vram_gb = (
    torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if torch.cuda.is_available()
    else 0.0
)

if VRAM_MODE == "auto":
    STAGE_OFFLOAD = total_vram_gb < 38.0
elif VRAM_MODE.startswith("stage-offload"):
    STAGE_OFFLOAD = True
else:
    STAGE_OFFLOAD = False

print(f"[INFO] Device: {DEVICE} ({total_vram_gb:.1f} GB VRAM) | STAGE_OFFLOAD: {STAGE_OFFLOAD}")

if "pipe" in globals() and pipe is not None:
    print("[INFO] Releasing previous pipeline instance ...")
    del pipe
    torch_gc()

t0 = time.perf_counter()
print("[1/6] Loading Tokenizer & UMT5-XXL Text Encoder (bfloat16) ...")
tokenizer = AutoTokenizer.from_pretrained(str(BASE_DIR), subfolder="tokenizer", torch_dtype=torch.bfloat16)
text_encoder = UMT5EncoderModel.from_pretrained(
    str(BASE_DIR), subfolder="text_encoder", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
)
text_encoder.eval()

print("[2/6] Loading Wan 2.1 3D Video VAE & FlowMatch Scheduler ...")
vae = AutoencoderKLWan.from_pretrained(str(BASE_DIR), subfolder="vae", torch_dtype=torch.bfloat16)
vae.eval()
scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(
    str(AVATAR_DIR), subfolder="scheduler", torch_dtype=torch.bfloat16
)

print("[3/6] Loading LongCat-Video-Avatar-1.5 3D DiT ...")
if USE_INT8:
    dit = load_quantized_dit_fast(str(AVATAR_DIR), subfolder="base_model_int8", cp_split_hw=[1, 1])
else:
    bf16_dir = AVATAR_DIR / "base_model"
    if not bf16_dir.exists():
        raise FileNotFoundError(
            f"{bf16_dir} not found. Enable DOWNLOAD_BASE_BF16_DIT in STEP 2 or set USE_INT8 = True."
        )
    dit = LongCatVideoAvatarTransformer3DModel.from_pretrained(
        str(AVATAR_DIR), subfolder="base_model", cp_split_hw=[1, 1], torch_dtype=torch.bfloat16
    )

if USE_DISTILL_LORA:
    lora_path = AVATAR_DIR / "lora" / "dmd_lora.safetensors"
    print(f"[4/6] Loading & activating DMD2 8-step distillation LoRA ({lora_path.name}) ...")
    dit.load_lora(str(lora_path), "dmd", multiplier=1.0, lora_network_dim=128, lora_network_alpha=64)
    dit.enable_loras(["dmd"])
    for lora_net in dit.lora_dict.values():
        for lora_mod in lora_net.loras:
            lora_mod.to(dtype=torch.bfloat16)
    gc.collect()
else:
    print("[4/6] Skipping DMD2 LoRA (full 50-step CFG mode)")

print("[5/6] Loading Whisper-Large-v3 Audio Encoder & Feature Extractor ...")
whisper_dir = str(AVATAR_DIR / "whisper-large-v3")
audio_encoder = get_audio_encoder(whisper_dir, "avatar-v1.5")
audio_encoder.to(dtype=torch.bfloat16).eval()
audio_feature_extractor = get_audio_feature_extractor(whisper_dir, "avatar-v1.5")

print("[6/6] Loading Kim_Vocal_2 ONNX Vocal Separator ...")
AUDIO_TEMP_DIR = Path("/tmp/longcat_audio_temp")
(AUDIO_TEMP_DIR / "vocals").mkdir(parents=True, exist_ok=True)
vocal_separator = Separator(
    output_dir=str(AUDIO_TEMP_DIR / "vocals"),
    output_single_stem="vocals",
    model_file_dir=str(AVATAR_DIR / "vocal_separator"),
)
vocal_separator.load_model("Kim_Vocal_2.onnx")

pipe = LongCatVideoAvatarPipeline(
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    vae=vae,
    scheduler=scheduler,
    dit=dit,
    audio_encoder=audio_encoder,
    audio_feature_extractor=audio_feature_extractor,
    model_type="avatar-v1.5",
)
pipe.device = DEVICE


# ---------------------------------------------------------------------------
# 4. Stage-Wise GPU VRAM Manager (L4 24GB Safe at 100% Native GPU Speed)
# ---------------------------------------------------------------------------
def _move_dit(dit_model, target_device):
    dit_model.to(target_device, non_blocking=True)
    for lora_net in getattr(dit_model, "lora_dict", {}).values():
        for lora_mod in lora_net.loras:
            lora_mod.to(target_device, dtype=torch.bfloat16, non_blocking=True)


def _configure_vram_placement(pipeline, stage_offload: bool):
    if not stage_offload:
        print("[VRAM Manager] Full-GPU mode: moving all components to CUDA ...")
        _move_dit(pipeline.dit, DEVICE)
        pipeline.text_encoder.to(DEVICE, dtype=torch.bfloat16, non_blocking=True)
        pipeline.vae.to(DEVICE, dtype=torch.bfloat16, non_blocking=True)
        pipeline.audio_encoder.to(DEVICE, dtype=torch.bfloat16, non_blocking=True)
        torch_gc()
        return

    print("[VRAM Manager] Stage-Offload mode active (L4 24GB optimized): staging components on demand ...")
    pipeline.text_encoder.to("cpu")
    pipeline.vae.to("cpu")
    pipeline.audio_encoder.to("cpu")
    _move_dit(pipeline.dit, "cpu")
    torch_gc()

    orig_get_audio_emb = pipeline.get_audio_embedding
    orig_text_forward = pipeline.text_encoder.forward
    orig_vae_encode = pipeline.vae.encode
    orig_vae_decode = pipeline.vae.decode
    orig_dit_forward = pipeline.dit.forward

    def _staged_get_audio_embedding(*args, **kwargs):
        pipeline.audio_encoder.to(DEVICE, dtype=torch.bfloat16)
        try:
            return orig_get_audio_emb(*args, **kwargs)
        finally:
            pipeline.audio_encoder.to("cpu")
            torch_gc()

    def _staged_text_forward(*args, **kwargs):
        if next(pipeline.dit.parameters()).device.type == "cuda":
            _move_dit(pipeline.dit, "cpu")
            torch_gc()
        pipeline.text_encoder.to(DEVICE, dtype=torch.bfloat16)
        try:
            return orig_text_forward(*args, **kwargs)
        finally:
            pipeline.text_encoder.to("cpu")
            torch_gc()

    def _staged_vae_encode(*args, **kwargs):
        pipeline.vae.to(DEVICE, dtype=torch.bfloat16)
        try:
            return orig_vae_encode(*args, **kwargs)
        finally:
            pipeline.vae.to("cpu")
            torch_gc()

    def _staged_dit_forward(*args, **kwargs):
        if next(pipeline.dit.parameters()).device.type != "cuda":
            pipeline.text_encoder.to("cpu")
            pipeline.audio_encoder.to("cpu")
            pipeline.vae.to("cpu")
            torch_gc()
            _move_dit(pipeline.dit, DEVICE)
        return orig_dit_forward(*args, **kwargs)

    def _staged_vae_decode(*args, **kwargs):
        if next(pipeline.dit.parameters()).device.type == "cuda":
            _move_dit(pipeline.dit, "cpu")
            torch_gc()
        pipeline.vae.to(DEVICE, dtype=torch.bfloat16)
        try:
            return orig_vae_decode(*args, **kwargs)
        finally:
            pipeline.vae.to("cpu")
            torch_gc()

    pipeline.get_audio_embedding = _staged_get_audio_embedding
    pipeline.text_encoder.forward = _staged_text_forward
    pipeline.vae.encode = _staged_vae_encode
    pipeline.dit.forward = _staged_dit_forward
    pipeline.vae.decode = _staged_vae_decode


_configure_vram_placement(pipe, STAGE_OFFLOAD)
print(f"\n[READY] Pipeline initialized in {time.perf_counter() - t0:.1f}s")


# ---------------------------------------------------------------------------
# 5. Unified Generation Engine (AI2V, Long-Video AVC, AT2V & Dual-Speaker)
# ---------------------------------------------------------------------------
DEFAULT_NEGATIVE_PROMPT = (
    "Close-up, bright tones, overexposed, static, blurred details, subtitles, style, works, "
    "paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, "
    "ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, "
    "misshapen limbs, fused fingers, still picture, messy background, three legs, many people in "
    "the background, walking backwards"
)

_VOCAL_CACHE = OrderedDict()
_AUDIO_EMB_CACHE = OrderedDict()
_AUDIO_CACHE_DIR = Path(tempfile.gettempdir()) / "longcat_cache" / "audio_emb"
_AUDIO_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _file_sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def _get_dbcache_config(accel_mode: str, use_distill: bool) -> dict:
    if accel_mode == "off":
        return dict(enabled=False)
    if use_distill:
        # 8-step DMD2 schedule has larger step-to-step residual deltas (~0.15-0.30) than 50-step CFG
        presets = {
            "balanced": dict(
                enabled=True, fn=1, bn=0, warmup_steps=1,
                max_cached_steps=2, max_continuous_cached_steps=1,
                residual_diff_threshold=0.35, downsample_factor=4,
            ),
            "fast": dict(
                enabled=True, fn=1, bn=0, warmup_steps=1,
                max_cached_steps=3, max_continuous_cached_steps=1,
                residual_diff_threshold=0.35, downsample_factor=4,
            ),
            "quality": dict(
                enabled=True, fn=1, bn=0, warmup_steps=2,
                max_cached_steps=1, max_continuous_cached_steps=1,
                residual_diff_threshold=0.20, downsample_factor=4,
            ),
        }
    else:
        presets = {
            "balanced": dict(
                enabled=True, fn=1, bn=0, warmup_steps=2,
                max_cached_steps=12, max_continuous_cached_steps=2,
                residual_diff_threshold=0.08, downsample_factor=4,
            ),
            "fast": dict(
                enabled=True, fn=1, bn=0, warmup_steps=2,
                max_cached_steps=18, max_continuous_cached_steps=3,
                residual_diff_threshold=0.12, downsample_factor=4,
            ),
            "quality": dict(
                enabled=True, fn=1, bn=0, warmup_steps=4,
                max_cached_steps=6, max_continuous_cached_steps=1,
                residual_diff_threshold=0.05, downsample_factor=4,
            ),
        }
    return presets.get(accel_mode, presets["balanced"])


def _extract_vocal(source_audio_path: str, vocal_mode: str = "separate") -> str:
    if not source_audio_path or vocal_mode == "raw":
        return source_audio_path
    try:
        audio_hash = _file_sha256(source_audio_path)
        cached = _VOCAL_CACHE.get(audio_hash)
        if cached and Path(cached).exists():
            return cached
        stable_path = AUDIO_TEMP_DIR / "vocals" / f"{audio_hash[:16]}_vocals.wav"
        if stable_path.exists():
            _VOCAL_CACHE[audio_hash] = str(stable_path)
            return str(stable_path)
        outputs = vocal_separator.separate(source_audio_path)
        if outputs:
            cand = (AUDIO_TEMP_DIR / "vocals" / outputs[0]).resolve()
            if cand.exists():
                shutil.copyfile(cand, stable_path)
                try:
                    os.remove(cand)
                except OSError:
                    pass
                _VOCAL_CACHE[audio_hash] = str(stable_path)
                return str(stable_path)
    except Exception as exc:
        print(f"  [WARN] Vocal separation failed ({exc}); using raw audio.")
    return source_audio_path


def _get_cached_full_audio_emb(raw_audio_path: str, target_duration: float, vocal_mode: str, save_fps: int, audio_stride: int, sr: int = 16000):
    audio_hash = _file_sha256(raw_audio_path)
    cache_key = (audio_hash, vocal_mode, round(float(target_duration), 3), int(save_fps), int(audio_stride))
    if cache_key in _AUDIO_EMB_CACHE:
        _AUDIO_EMB_CACHE.move_to_end(cache_key)
        print(f"  [Cache] Audio embedding RAM hit ({audio_hash[:10]})")
        return _AUDIO_EMB_CACHE[cache_key].to(DEVICE)

    key_json = json.dumps(cache_key, sort_keys=True, separators=(",", ":"))
    cache_file = _AUDIO_CACHE_DIR / f"{hashlib.sha256(key_json.encode('utf-8')).hexdigest()}.pt"
    if cache_file.exists():
        try:
            emb_cpu = torch.load(cache_file, map_location="cpu")
            _AUDIO_EMB_CACHE[cache_key] = emb_cpu
            print(f"  [Cache] Audio embedding disk hit ({audio_hash[:10]})")
            return emb_cpu.to(DEVICE)
        except Exception:
            pass

    vocal_path = _extract_vocal(raw_audio_path, vocal_mode=vocal_mode)
    speech, _ = librosa.load(vocal_path, sr=sr)
    added_samples = math.ceil((target_duration - (len(speech) / float(sr))) * sr)
    if added_samples > 0:
        speech = np.append(speech, np.zeros(added_samples, dtype=speech.dtype))

    full_emb = pipe.get_audio_embedding(
        speech, fps=save_fps * audio_stride, device=DEVICE, sample_rate=sr, model_type="avatar-v1.5"
    )
    if torch.isnan(full_emb).any():
        raise ValueError("Audio embedding contains NaN values — please check the input audio file.")

    emb_cpu = full_emb.detach().cpu()
    _AUDIO_EMB_CACHE[cache_key] = emb_cpu
    while len(_AUDIO_EMB_CACHE) > 8:
        _AUDIO_EMB_CACHE.popitem(last=False)
    try:
        torch.save(emb_cpu, cache_file)
    except Exception:
        pass
    return full_emb


def _slice_audio_window(full_audio_emb: torch.Tensor, start_idx: int, num_frames: int, audio_stride: int = 1):
    indices = torch.arange(2 * 2 + 1, device=full_audio_emb.device) - 2
    end_idx = start_idx + audio_stride * num_frames
    center_indices = torch.arange(start_idx, end_idx, audio_stride, device=full_audio_emb.device).unsqueeze(1) + indices.unsqueeze(0)
    center_indices = torch.clamp(center_indices, min=0, max=full_audio_emb.shape[0] - 1)
    return full_audio_emb[center_indices][None, ...].to(DEVICE), end_idx


def _build_multi_masks(src_width: int, src_height: int, bbox_1=None, bbox_2=None):
    background_mask = torch.zeros([src_height, src_width], dtype=torch.float32)
    human_mask1 = torch.zeros([src_height, src_width], dtype=torch.float32)
    human_mask2 = torch.zeros([src_height, src_width], dtype=torch.float32)

    if bbox_1 is None or bbox_2 is None:
        face_scale = 0.1
        y_min, y_max = int(src_height * face_scale), int(src_height * (1.0 - face_scale))
        half_w = src_width // 2
        l_xmin, l_xmax = int(half_w * face_scale), int(half_w * (1.0 - face_scale))
        r_xmin, r_xmax = int(half_w * face_scale + half_w), int(half_w * (1.0 - face_scale) + half_w)
        bbox_1 = [y_min, l_xmin, y_max, l_xmax]
        bbox_2 = [y_min, r_xmin, y_max, r_xmax]

    ly1, lx1, ly2, lx2 = [int(v) for v in bbox_1]
    ry1, rx1, ry2, rx2 = [int(v) for v in bbox_2]
    human_mask1[ly1:ly2, lx1:lx2] = 1.0
    human_mask2[ry1:ry2, rx1:rx2] = 1.0
    background_mask = torch.where((human_mask1 + human_mask2) > 0, torch.tensor(0.0), torch.tensor(1.0))
    return torch.stack([human_mask1, human_mask2, background_mask], dim=0).to(DEVICE)


def _save_video_ffmpeg_fast(frames_pil: list, out_base: str, audio_path: str, fps: int = 25, quality: int = 5) -> str:
    """Single-pass H.264 encode + stream-copy AAC audio mux (avoids double video re-encoding)."""
    temp_video = out_base + "-tempvideo.mp4"
    out_mp4 = out_base + ".mp4"
    os.makedirs(os.path.dirname(out_mp4), exist_ok=True)

    writer = imageio.get_writer(temp_video, fps=fps, codec="libx264", quality=quality)
    try:
        for frame in frames_pil:
            writer.append_data(np.asarray(frame))
    finally:
        writer.close()

    duration = len(frames_pil) / float(fps)
    cmd = [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", temp_video,
        "-i", audio_path,
        "-t", f"{duration:.3f}",
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "128k",
        "-shortest",
        out_mp4,
    ]
    res = subprocess.run(cmd)
    try:
        os.remove(temp_video)
    except OSError:
        pass
    if res.returncode != 0:
        output_tensor = torch.from_numpy(np.stack([np.asarray(f) for f in frames_pil], axis=0))
        save_video_ffmpeg(output_tensor, out_base, audio_path, fps=fps, quality=quality)
        del output_tensor
    return out_mp4


def generate_avatar_video(
    mode: str = "ai2v",
    image_path: str = None,
    audio_path: str = None,
    prompt: str = "A person is speaking naturally in front of a steady camera, realistic facial expressions.",
    negative_prompt: str = DEFAULT_NEGATIVE_PROMPT,
    resolution: str = "480p",
    num_frames: int = 93,
    num_cond_frames: int = 13,
    num_segments: int = 1,
    num_inference_steps: int = None,
    text_guidance_scale: float = None,
    audio_guidance_scale: float = None,
    vocal_mode: str = "separate",
    accel_mode: str = "balanced",
    seed: int = 42,
    ref_img_index: int = 10,
    mask_frame_range: int = 3,
    resize_mode: str = "crop",
    # Dual-speaker parameters
    audio_path_2: str = None,
    audio_type: str = "add",
    bbox_1: list = None,
    bbox_2: list = None,
    output_stem: str = None,
) -> str:
    """Run LongCat-Video-Avatar-1.5 generation and save MP4 to OUT_DIR."""
    t_start = time.perf_counter()
    save_fps = 25
    audio_stride = 1
    sr = 16000

    use_distill = bool(USE_DISTILL_LORA)
    if num_inference_steps is None:
        num_inference_steps = 8 if use_distill else 50
    if text_guidance_scale is None:
        text_guidance_scale = 1.0 if use_distill else 4.0
    if audio_guidance_scale is None:
        audio_guidance_scale = 1.0 if use_distill else 4.0

    if (num_frames - 1) % 4 != 0:
        num_frames = ((num_frames - 1) // 4) * 4 + 1
    num_frames = max(21, int(num_frames))

    db_cfg = _get_dbcache_config(accel_mode, use_distill=use_distill)
    pipe.dit.configure_dbcache(**db_cfg)

    if seed is None or int(seed) < 0:
        seed = random.randint(0, 2**31 - 1)
    seed = int(seed)
    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(seed)

    seg1_dur = num_frames / float(save_fps)
    seg_step_dur = (num_frames - num_cond_frames) / float(save_fps)

    if mode == "multi":
        if not audio_path and not audio_path_2:
            raise ValueError("Dual-speaker mode requires at least one audio path (audio_path or audio_path_2).")
        dur1 = librosa.get_duration(path=audio_path) if audio_path else 0.0
        dur2 = librosa.get_duration(path=audio_path_2) if audio_path_2 else 0.0
        raw_total_dur = (dur1 + dur2) if audio_type == "add" else max(dur1, dur2)
    else:
        if not audio_path:
            raise ValueError("audio_path is required.")
        raw_total_dur = librosa.get_duration(path=audio_path)

    if int(num_segments) <= 0:
        if raw_total_dur <= seg1_dur or mode == "at2v":
            num_segments = 1
        else:
            num_segments = 1 + math.ceil((raw_total_dur - seg1_dur) / seg_step_dur)
        print(f"  [Auto-Segments] Audio duration {raw_total_dur:.2f}s -> {num_segments} segment(s)")
    else:
        num_segments = max(1, int(num_segments))
        if mode == "at2v" and num_segments > 1:
            print("  [INFO] AT2V (no reference image) generates a single segment; setting num_segments=1.")
            num_segments = 1

    generate_duration = seg1_dur + (num_segments - 1) * seg_step_dur
    mux_audio_path = audio_path
    temp_mux_wav = None

    # 1. Extract audio embeddings via Whisper-Large-v3
    print(f"  [Stage 1/3] Preparing audio embeddings (mode={mode}, duration={generate_duration:.2f}s, vocal={vocal_mode}) ...")
    if mode == "multi":
        v1 = _extract_vocal(audio_path, vocal_mode) if audio_path else None
        v2 = _extract_vocal(audio_path_2, vocal_mode) if audio_path_2 else None
        s1, _ = librosa.load(v1, sr=sr) if v1 else (None, sr)
        r1, _ = librosa.load(audio_path, sr=sr) if audio_path else (None, sr)
        s2, _ = librosa.load(v2, sr=sr) if v2 else (None, sr)
        r2, _ = librosa.load(audio_path_2, sr=sr) if audio_path_2 else (None, sr)
        if s1 is None:
            s1 = np.zeros_like(s2)
            r1 = np.zeros_like(r2)
        if s2 is None:
            s2 = np.zeros_like(s1)
            r2 = np.zeros_like(r1)
        if audio_type == "add":
            left_ext = np.concatenate([s1, np.zeros_like(s2)])
            right_ext = np.concatenate([np.zeros_like(s1), s2])
            merge_raw = np.concatenate([r1, np.zeros_like(r2)]) + np.concatenate([np.zeros_like(r1), r2])
        else:
            max_len = max(len(s1), len(s2))
            if len(s1) < max_len:
                s1 = np.pad(s1, (0, max_len - len(s1)))
                r1 = np.pad(r1, (0, max_len - len(r1)))
            if len(s2) < max_len:
                s2 = np.pad(s2, (0, max_len - len(s2)))
                r2 = np.pad(r2, (0, max_len - len(r2)))
            left_ext, right_ext = s1, s2
            merge_raw = r1 + r2

        added = math.ceil((generate_duration - (len(left_ext) / float(sr))) * sr)
        if added > 0:
            left_ext = np.append(left_ext, np.zeros(added, dtype=left_ext.dtype))
            right_ext = np.append(right_ext, np.zeros(added, dtype=right_ext.dtype))

        temp_mux_wav = str(AUDIO_TEMP_DIR / f"merge_{int(time.time()*1000)}.wav")
        sf.write(temp_mux_wav, merge_raw, sr)
        mux_audio_path = temp_mux_wav

        left_full_emb = pipe.get_audio_embedding(left_ext, fps=save_fps * audio_stride, device=DEVICE, sample_rate=sr, model_type="avatar-v1.5")
        right_full_emb = pipe.get_audio_embedding(right_ext, fps=save_fps * audio_stride, device=DEVICE, sample_rate=sr, model_type="avatar-v1.5")
        l_slice, _ = _slice_audio_window(left_full_emb, 0, num_frames, audio_stride)
        r_slice, _ = _slice_audio_window(right_full_emb, 0, num_frames, audio_stride)
        audio_emb_seg1 = torch.cat([l_slice, r_slice], dim=0)
    else:
        full_audio_emb = _get_cached_full_audio_emb(
            audio_path, generate_duration, vocal_mode=vocal_mode, save_fps=save_fps, audio_stride=audio_stride, sr=sr
        )
        audio_emb_seg1, _ = _slice_audio_window(full_audio_emb, 0, num_frames, audio_stride)

    # 2. Segment 1 Generation
    pipe.dit.reset_dbcache()
    print(f"  [Stage 2/3] Generating Segment 1/{num_segments} ({resolution}, {num_frames} frames, {num_inference_steps} steps, seed={seed}) ...")

    ref_target_masks = None
    with torch.inference_mode():
        if mode == "at2v":
            height, width = (480, 832) if resolution == "480p" else (768, 1280)
            output_video = pipe.generate_at2v(
                prompt=prompt,
                negative_prompt=negative_prompt,
                height=height,
                width=width,
                num_frames=num_frames,
                num_inference_steps=num_inference_steps,
                text_guidance_scale=text_guidance_scale,
                audio_guidance_scale=audio_guidance_scale,
                generator=generator,
                output_type="np",
                audio_emb=audio_emb_seg1,
                use_distill=use_distill,
            )
            frames_np = output_video[0]
            all_pil_frames = [PIL.Image.fromarray((frames_np[i] * 255).astype(np.uint8)) for i in range(frames_np.shape[0])]
            latent = None
        else:
            if not image_path:
                raise ValueError(f"image_path is required for mode='{mode}'.")
            img = load_image(image_path) if isinstance(image_path, (str, Path)) else image_path.convert("RGB")
            if mode == "multi":
                src_w, src_h = img.size
                ref_target_masks = _build_multi_masks(src_w, src_h, bbox_1=bbox_1, bbox_2=bbox_2)

            output_tuple = pipe.generate_ai2v(
                image=img,
                prompt=prompt,
                negative_prompt=negative_prompt,
                resolution=resolution,
                num_frames=num_frames,
                num_inference_steps=num_inference_steps,
                text_guidance_scale=text_guidance_scale,
                audio_guidance_scale=audio_guidance_scale,
                output_type="both",
                generator=generator,
                audio_emb=audio_emb_seg1,
                ref_target_masks=ref_target_masks,
                use_distill=use_distill,
                resize_mode=resize_mode,
            )
            output_video, latent = output_tuple
            frames_np = output_video[0]
            all_pil_frames = [PIL.Image.fromarray((frames_np[i] * 255).astype(np.uint8)) for i in range(frames_np.shape[0])]
            del output_video

        db_stats = pipe.dit.get_dbcache_stats()
        if db_stats.get("steps"):
            print(f"    DBCache stats (Seg 1): cached {len(db_stats.get('cached_steps', []))}/{db_stats['steps']} steps (diffs={db_stats.get('diffs', [])})")
        torch_gc()

        # 3. Multi-Segment Video Continuation (AVC) if num_segments > 1
        if num_segments > 1 and latent is not None:
            width, height = all_pil_frames[0].size
            current_video = list(all_pil_frames)
            ref_latent = latent[:, :, :1].clone()
            audio_start_idx = 0

            for seg_idx in range(1, num_segments):
                print(f"  [Stage 2/3] Continuing Segment {seg_idx + 1}/{num_segments} via AVC (overlap={num_cond_frames}f, ref_idx={ref_img_index}) ...")
                audio_start_idx = audio_start_idx + audio_stride * (num_frames - num_cond_frames)
                if mode == "multi":
                    l_slice, _ = _slice_audio_window(left_full_emb, audio_start_idx, num_frames, audio_stride)
                    r_slice, _ = _slice_audio_window(right_full_emb, audio_start_idx, num_frames, audio_stride)
                    audio_emb_next = torch.cat([l_slice, r_slice], dim=0)
                else:
                    audio_emb_next, _ = _slice_audio_window(full_audio_emb, audio_start_idx, num_frames, audio_stride)

                pipe.dit.reset_dbcache()
                output_tuple = pipe.generate_avc(
                    video=current_video,
                    video_latent=latent,
                    prompt=prompt,
                    negative_prompt=negative_prompt,
                    height=height,
                    width=width,
                    num_frames=num_frames,
                    num_cond_frames=num_cond_frames,
                    num_inference_steps=num_inference_steps,
                    text_guidance_scale=text_guidance_scale,
                    audio_guidance_scale=audio_guidance_scale,
                    generator=generator,
                    output_type="both",
                    use_kv_cache=True,
                    offload_kv_cache=STAGE_OFFLOAD,
                    enhance_hf=not use_distill,
                    audio_emb=audio_emb_next,
                    ref_latent=ref_latent,
                    ref_img_index=ref_img_index,
                    mask_frame_range=mask_frame_range,
                    ref_target_masks=ref_target_masks,
                    use_distill=use_distill,
                    resize_mode=resize_mode,
                )
                output_video, latent = output_tuple
                frames_np = output_video[0]
                new_pil_frames = [PIL.Image.fromarray((frames_np[i] * 255).astype(np.uint8)) for i in range(frames_np.shape[0])]
                del output_video
                all_pil_frames.extend(new_pil_frames[num_cond_frames:])
                current_video = new_pil_frames
                torch_gc()

    # 4. Trim frames to actual audio length if multi-segment padded past the end of speech
    max_audio_frames = max(1, int(round(raw_total_dur * save_fps)))
    if num_segments > 1 and len(all_pil_frames) > max_audio_frames:
        all_pil_frames = all_pil_frames[:max_audio_frames]

    # 5. Mux video + original audio via single-pass FFmpeg
    ts_str = time.strftime("%Y%m%d_%H%M%S")
    stem = output_stem or f"LongCatAvatar15_{mode}_{resolution}_{len(all_pil_frames)}f_seed{seed}_{ts_str}"
    out_base = str(OUT_DIR / stem)

    print(f"  [Stage 3/3] Encoding {len(all_pil_frames)} frames @ {save_fps} FPS -> {out_base}.mp4 ...")
    out_mp4 = _save_video_ffmpeg_fast(all_pil_frames, out_base, mux_audio_path, fps=save_fps, quality=5)

    if temp_mux_wav and os.path.exists(temp_mux_wav):
        try:
            os.remove(temp_mux_wav)
        except OSError:
            pass

    elapsed = time.perf_counter() - t_start
    print(f"  [DONE] Saved {out_mp4} in {elapsed:.1f}s ({len(all_pil_frames)/ float(save_fps):.2f}s video)")
    return out_mp4

In [ ]:
#@title STEP 4 — Launch Gradio Studio UI (Single Avatar, AT2V & Dual-Speaker)
import gradio as gr
from pathlib import Path

if "generate_avatar_video" not in globals():
    raise RuntimeError("Please run STEP 3 first to load the LongCat-Video-Avatar-1.5 pipeline.")

EX_SINGLE_DIR = REPO_DIR / "assets/avatar/single"
EX_EXAMPLES_DIR = REPO_DIR / "assets/avatar/examples"
EX_MULTI_DIR = REPO_DIR / "assets/avatar/multi"


def _ui_single_avatar(
    image_path,
    audio_path,
    prompt,
    resolution,
    num_frames,
    num_segments,
    vocal_mode_label,
    accel_label,
    seed,
    ref_img_index,
    mask_frame_range,
):
    vocal_mode = "separate" if "Separate" in vocal_mode_label else "raw"
    accel_map = {
        "Balanced (DBCache)": "balanced",
        "Quality (Conservative DBCache)": "quality",
        "Fast (Aggressive DBCache)": "fast",
        "Off (No DBCache)": "off",
    }
    out_mp4 = generate_avatar_video(
        mode="ai2v",
        image_path=image_path,
        audio_path=audio_path,
        prompt=prompt,
        resolution=resolution,
        num_frames=int(num_frames),
        num_segments=int(num_segments),
        vocal_mode=vocal_mode,
        accel_mode=accel_map.get(accel_label, "balanced"),
        seed=int(seed),
        ref_img_index=int(ref_img_index),
        mask_frame_range=int(mask_frame_range),
    )
    return out_mp4, f"Saved to Google Drive: {out_mp4}"


def _ui_at2v(
    audio_path,
    prompt,
    resolution,
    num_frames,
    vocal_mode_label,
    accel_label,
    seed,
):
    vocal_mode = "separate" if "Separate" in vocal_mode_label else "raw"
    accel_map = {
        "Balanced (DBCache)": "balanced",
        "Quality (Conservative DBCache)": "quality",
        "Fast (Aggressive DBCache)": "fast",
        "Off (No DBCache)": "off",
    }
    out_mp4 = generate_avatar_video(
        mode="at2v",
        audio_path=audio_path,
        prompt=prompt,
        resolution=resolution,
        num_frames=int(num_frames),
        num_segments=1,
        vocal_mode=vocal_mode,
        accel_mode=accel_map.get(accel_label, "balanced"),
        seed=int(seed),
    )
    return out_mp4, f"Saved to Google Drive: {out_mp4}"


def _ui_multi_speaker(
    image_path,
    audio_path_1,
    audio_path_2,
    audio_type_label,
    prompt,
    resolution,
    num_frames,
    num_segments,
    bbox_1_str,
    bbox_2_str,
    vocal_mode_label,
    accel_label,
    seed,
):
    vocal_mode = "separate" if "Separate" in vocal_mode_label else "raw"
    audio_type = "add" if "Sequential" in audio_type_label else "para"
    accel_map = {
        "Balanced (DBCache)": "balanced",
        "Quality (Conservative DBCache)": "quality",
        "Fast (Aggressive DBCache)": "fast",
        "Off (No DBCache)": "off",
    }

    def _parse_bbox(s):
        if not s or not s.strip():
            return None
        parts = [int(float(x.strip())) for x in s.split(",") if x.strip()]
        return parts if len(parts) == 4 else None

    out_mp4 = generate_avatar_video(
        mode="multi",
        image_path=image_path,
        audio_path=audio_path_1,
        audio_path_2=audio_path_2,
        audio_type=audio_type,
        bbox_1=_parse_bbox(bbox_1_str),
        bbox_2=_parse_bbox(bbox_2_str),
        prompt=prompt,
        resolution=resolution,
        num_frames=int(num_frames),
        num_segments=int(num_segments),
        vocal_mode=vocal_mode,
        accel_mode=accel_map.get(accel_label, "balanced"),
        seed=int(seed),
    )
    return out_mp4, f"Saved to Google Drive: {out_mp4}"


with gr.Blocks(title="LongCat-Video-Avatar 1.5 Studio") as demo:
    gr.Markdown(
        "## LongCat-Video-Avatar 1.5 — Audio-Driven Avatar Studio\n"
        "All outputs are automatically saved to your Google Drive `OUT_DIR` (`" + str(OUT_DIR) + "`)."
    )

    with gr.Tabs():
        # TAB 1: Single Avatar AI2V + Long-Video AVC
        with gr.Tab("1. Single Avatar (Image + Audio -> Video)"):
            with gr.Row():
                with gr.Column(scale=1):
                    s_img = gr.Image(type="filepath", label="Reference Portrait / Character Image")
                    s_aud = gr.Audio(type="filepath", label="Driving Speech / Singing Audio")
                    s_prompt = gr.Textbox(
                        label="Scene & Motion Prompt",
                        lines=3,
                        value="A person is speaking naturally to the camera under soft studio lighting, expressive facial movements and natural head gestures.",
                    )
                    with gr.Row():
                        s_res = gr.Dropdown(["480p", "720p"], value="480p", label="Resolution")
                        s_frames = gr.Slider(49, 125, value=93, step=4, label="Frames per Segment (93 = 3.7s @ 25fps)")
                    with gr.Row():
                        s_segs = gr.Slider(
                            0, 10, value=1, step=1,
                            label="Segments (0 = Auto-fit Full Audio, 1 = Single Clip, >1 = Multi-Segment AVC)"
                        )
                        s_seed = gr.Number(value=42, precision=0, label="Seed (-1 = Random)")
                    with gr.Accordion("Advanced Audio, DBCache & Continuation Settings", open=False):
                        s_vocal = gr.Radio(
                            ["Separate Vocals (Kim_Vocal_2 ONNX)", "Raw Audio (Skip Separator)"],
                            value="Separate Vocals (Kim_Vocal_2 ONNX)",
                            label="Audio Preprocessing",
                        )
                        s_accel = gr.Dropdown(
                            ["Balanced (DBCache)", "Quality (Conservative DBCache)", "Fast (Aggressive DBCache)", "Off (No DBCache)"],
                            value="Balanced (DBCache)",
                            label="DBCache Acceleration",
                        )
                        with gr.Row():
                            s_ref_idx = gr.Slider(0, 24, value=10, step=1, label="AVC ref_img_index")
                            s_mask_rng = gr.Slider(0, 8, value=3, step=1, label="AVC mask_frame_range")
                    s_btn = gr.Button("Generate Avatar Video", variant="primary")

                with gr.Column(scale=1):
                    s_vid = gr.Video(label="Generated Avatar Video (25 FPS)")
                    s_status = gr.Textbox(label="Output Path", interactive=False)

            single_examples = []
            if (EX_EXAMPLES_DIR / "photoreal_person.png").exists() and (EX_EXAMPLES_DIR / "audio/photoreal_person_voice.wav").exists():
                single_examples.append([
                    str(EX_EXAMPLES_DIR / "photoreal_person.png"),
                    str(EX_EXAMPLES_DIR / "audio/photoreal_person_voice.wav"),
                    "A photorealistic person speaking naturally to the camera in a clean studio portrait, soft neutral background, realistic facial expression.",
                ])
            if (EX_EXAMPLES_DIR / "orc_warrior.png").exists() and (EX_EXAMPLES_DIR / "audio/orc_warrior_voice.wav").exists():
                single_examples.append([
                    str(EX_EXAMPLES_DIR / "orc_warrior.png"),
                    str(EX_EXAMPLES_DIR / "audio/orc_warrior_voice.wav"),
                    "A fantasy orc warrior portrait speaking directly to the camera, cinematic studio lighting, detailed armor, expressive face, smoky neutral backdrop.",
                ])
            if (EX_SINGLE_DIR / "character.png").exists() and (EX_EXAMPLES_DIR / "audio/character_voice.wav").exists():
                single_examples.append([
                    str(EX_SINGLE_DIR / "character.png"),
                    str(EX_EXAMPLES_DIR / "audio/character_voice.wav"),
                    "A friendly cartoon character with short brown hair speaking to the camera against a soft blue background, simple flat illustration style.",
                ])
            if single_examples:
                gr.Examples(
                    examples=single_examples,
                    inputs=[s_img, s_aud, s_prompt],
                    label="Built-in Single-Avatar Examples",
                )

            s_btn.click(
                fn=_ui_single_avatar,
                inputs=[s_img, s_aud, s_prompt, s_res, s_frames, s_segs, s_vocal, s_accel, s_seed, s_ref_idx, s_mask_rng],
                outputs=[s_vid, s_status],
            )

        # TAB 2: Audio + Text to Video (AT2V)
        with gr.Tab("2. Audio + Text Only (AT2V — No Reference Image)"):
            with gr.Row():
                with gr.Column(scale=1):
                    t_aud = gr.Audio(type="filepath", label="Driving Speech / Singing Audio")
                    t_prompt = gr.Textbox(
                        label="Character & Scene Description Prompt",
                        lines=3,
                        value="Medium shot of a charismatic podcast host wearing studio headphones in a neon-lit studio, speaking passionately into a broadcast microphone.",
                    )
                    with gr.Row():
                        t_res = gr.Dropdown(["480p", "720p"], value="480p", label="Resolution")
                        t_frames = gr.Slider(49, 125, value=93, step=4, label="Frames (93 = 3.7s @ 25fps)")
                        t_seed = gr.Number(value=42, precision=0, label="Seed (-1 = Random)")
                    t_vocal = gr.Radio(
                        ["Separate Vocals (Kim_Vocal_2 ONNX)", "Raw Audio (Skip Separator)"],
                        value="Separate Vocals (Kim_Vocal_2 ONNX)",
                        label="Audio Preprocessing",
                    )
                    t_accel = gr.Dropdown(
                        ["Balanced (DBCache)", "Quality (Conservative DBCache)", "Fast (Aggressive DBCache)", "Off (No DBCache)"],
                        value="Balanced (DBCache)",
                        label="DBCache Acceleration",
                    )
                    t_btn = gr.Button("Generate AT2V Video", variant="primary")
                with gr.Column(scale=1):
                    t_vid = gr.Video(label="Generated AT2V Video (25 FPS)")
                    t_status = gr.Textbox(label="Output Path", interactive=False)

            t_btn.click(
                fn=_ui_at2v,
                inputs=[t_aud, t_prompt, t_res, t_frames, t_vocal, t_accel, t_seed],
                outputs=[t_vid, t_status],
            )

        # TAB 3: Dual-Speaker Conversation
        with gr.Tab("3. Dual-Speaker Conversation (Multi-Person AI2V)"):
            with gr.Row():
                with gr.Column(scale=1):
                    m_img = gr.Image(type="filepath", label="Two-Person Reference Image")
                    with gr.Row():
                        m_aud1 = gr.Audio(type="filepath", label="Speaker 1 (Left) Audio")
                        m_aud2 = gr.Audio(type="filepath", label="Speaker 2 (Right) Audio")
                    m_atype = gr.Radio(
                        ["Sequential Turn-Taking (add)", "Simultaneous / Duet (para)"],
                        value="Sequential Turn-Taking (add)",
                        label="Multi-Audio Combination Mode",
                    )
                    m_prompt = gr.Textbox(
                        label="Dual-Speaker Scene Prompt",
                        lines=3,
                        value="Static camera, two people are sitting side by side in a warmly lit cafe, engaged in a lively conversation with natural facial expressions.",
                    )
                    with gr.Row():
                        m_res = gr.Dropdown(["480p", "720p"], value="480p", label="Resolution")
                        m_frames = gr.Slider(49, 125, value=93, step=4, label="Frames per Segment")
                        m_segs = gr.Slider(0, 10, value=1, step=1, label="Segments (0 = Auto-fit Audio)")
                        m_seed = gr.Number(value=42, precision=0, label="Seed")
                    with gr.Accordion("Optional Bounding Boxes & DBCache Settings", open=False):
                        gr.Markdown("Leave bounding boxes blank to automatically split the image into Left & Right speaker regions, or specify `y_min, x_min, y_max, x_max` in pixels.")
                        with gr.Row():
                            m_bbox1 = gr.Textbox(label="Speaker 1 BBox (y1, x1, y2, x2)", placeholder="Auto left half")
                            m_bbox2 = gr.Textbox(label="Speaker 2 BBox (y1, x1, y2, x2)", placeholder="Auto right half")
                        m_vocal = gr.Radio(
                            ["Separate Vocals (Kim_Vocal_2 ONNX)", "Raw Audio (Skip Separator)"],
                            value="Separate Vocals (Kim_Vocal_2 ONNX)",
                            label="Audio Preprocessing",
                        )
                        m_accel = gr.Dropdown(
                            ["Balanced (DBCache)", "Quality (Conservative DBCache)", "Fast (Aggressive DBCache)", "Off (No DBCache)"],
                            value="Balanced (DBCache)",
                            label="DBCache Acceleration",
                        )
                    m_btn = gr.Button("Generate Dual-Speaker Video", variant="primary")
                with gr.Column(scale=1):
                    m_vid = gr.Video(label="Generated Dual-Speaker Video (25 FPS)")
                    m_status = gr.Textbox(label="Output Path", interactive=False)

            multi_examples = []
            if (EX_MULTI_DIR / "introduce.png").exists() and (EX_MULTI_DIR / "introduce_woman.mp3").exists():
                multi_examples.append([
                    str(EX_MULTI_DIR / "introduce.png"),
                    str(EX_MULTI_DIR / "introduce_woman.mp3"),
                    str(EX_MULTI_DIR / "introduce_man.mp3"),
                    "Sequential Turn-Taking (add)",
                    "Static camera, two people are speaking. Two people sit side by side in a warmly lit cafe, engaged in a lively conversation.",
                    "100, 80, 800, 640",
                    "50, 720, 820, 1300",
                ])
            if (EX_MULTI_DIR / "sing.png").exists() and (EX_MULTI_DIR / "sing_man.WAV").exists():
                multi_examples.append([
                    str(EX_MULTI_DIR / "sing.png"),
                    str(EX_MULTI_DIR / "sing_man.WAV"),
                    str(EX_MULTI_DIR / "sing_woman.WAV"),
                    "Simultaneous / Duet (para)",
                    "Static camera, In a professional recording studio, two people stand facing each other, both wearing large headphones, singing into a condenser microphone.",
                    "",
                    "",
                ])
            if multi_examples:
                gr.Examples(
                    examples=multi_examples,
                    inputs=[m_img, m_aud1, m_aud2, m_atype, m_prompt, m_bbox1, m_bbox2],
                    label="Built-in Dual-Speaker Examples",
                )

            m_btn.click(
                fn=_ui_multi_speaker,
                inputs=[m_img, m_aud1, m_aud2, m_atype, m_prompt, m_res, m_frames, m_segs, m_bbox1, m_bbox2, m_vocal, m_accel, m_seed],
                outputs=[m_vid, m_status],
            )

demo.queue().launch(share=True, inline=True, show_error=True)

In [ ]:
#@title STEP 5 — Session Status, GPU Monitor & Optional Keep-Alive
RUN_KEEP_ALIVE_LOOP = False #@param {type:"boolean"}
KEEP_ALIVE_MINUTES = 60 #@param {type:"integer"}

import time
import torch
from pathlib import Path

print("=== LongCat-Video-Avatar 1.5 Session Status ===")
if torch.cuda.is_available():
    alloc_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
    res_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)
    tot_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"  GPU VRAM   : {alloc_gb:.2f} GB allocated / {res_gb:.2f} GB reserved / {tot_gb:.2f} GB total")
    print(f"  Stage Mode : {'STAGE_OFFLOAD (L4 24GB safe)' if globals().get('STAGE_OFFLOAD', True) else 'FULL_GPU (Resident)'}")

if "OUT_DIR" in globals() and Path(OUT_DIR).exists():
    mp4s = sorted(Path(OUT_DIR).glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
    print(f"  Output Dir : {OUT_DIR} ({len(mp4s)} MP4 file(s))")
    for p in mp4s[:10]:
        sz_mb = p.stat().st_size / (1024 ** 2)
        print(f"    - {p.name} ({sz_mb:.2f} MB)")

if RUN_KEEP_ALIVE_LOOP:
    print(f"\n[Keep-Alive] Holding cell execution for {KEEP_ALIVE_MINUTES} minutes (interrupt cell to stop) ...")
    end_time = time.time() + KEEP_ALIVE_MINUTES * 60
    try:
        while time.time() < end_time:
            rem = int((end_time - time.time()) / 60)
            print(f"  [Keep-Alive] Active — {rem} min remaining ...", end="\r", flush=True)
            time.sleep(30)
    except KeyboardInterrupt:
        print("\n[Keep-Alive] Stopped by user.")

In [ ]:
#@title STEP 6 — Direct Cell Generation (Single / Long-Video / AT2V / Dual-Speaker)
MODE = "ai2v (Image + Audio)" #@param ["ai2v (Image + Audio)", "at2v (Audio + Text Only)", "multi (Dual-Speaker Conversation)"]
IMAGE_PATH = "/content/LongCat-Video-Avatar-1.5/assets/avatar/examples/photoreal_person.png" #@param {type:"string"}
AUDIO_PATH = "/content/LongCat-Video-Avatar-1.5/assets/avatar/examples/audio/photoreal_person_voice.wav" #@param {type:"string"}
AUDIO_PATH_2 = "" #@param {type:"string"}
MULTI_AUDIO_TYPE = "add (Sequential Turn-Taking)" #@param ["add (Sequential Turn-Taking)", "para (Simultaneous / Duet)"]
PROMPT = "A photorealistic person speaking naturally to the camera in a clean studio portrait, soft neutral background, realistic facial expression." #@param {type:"string"}
RESOLUTION = "480p" #@param ["480p", "720p"]
FRAMES_PER_SEGMENT = 93 #@param {type:"slider", min:49, max:125, step:4}
NUM_SEGMENTS = 1 #@param {type:"slider", min:0, max:10, step:1}
VOCAL_MODE = "separate" #@param ["separate", "raw"]
ACCELERATION = "balanced" #@param ["balanced", "quality", "fast", "off"]
SEED = 42 #@param {type:"integer"}
REF_IMG_INDEX = 10 #@param {type:"slider", min:0, max:24, step:1}
MASK_FRAME_RANGE = 3 #@param {type:"slider", min:0, max:8, step:1}
DISPLAY_INLINE_VIDEO = True #@param {type:"boolean"}

from IPython.display import Video, display, FileLink

if "generate_avatar_video" not in globals():
    raise RuntimeError("Please run STEP 3 first to load the LongCat-Video-Avatar-1.5 pipeline.")

mode_key = MODE.split()[0].strip()
audio_type_key = MULTI_AUDIO_TYPE.split()[0].strip()

out_mp4_path = generate_avatar_video(
    mode=mode_key,
    image_path=IMAGE_PATH.strip() or None,
    audio_path=AUDIO_PATH.strip() or None,
    audio_path_2=AUDIO_PATH_2.strip() or None,
    audio_type=audio_type_key,
    prompt=PROMPT,
    resolution=RESOLUTION,
    num_frames=int(FRAMES_PER_SEGMENT),
    num_segments=int(NUM_SEGMENTS),
    vocal_mode=VOCAL_MODE,
    accel_mode=ACCELERATION,
    seed=int(SEED),
    ref_img_index=int(REF_IMG_INDEX),
    mask_frame_range=int(MASK_FRAME_RANGE),
)

display(FileLink(out_mp4_path))
if DISPLAY_INLINE_VIDEO:
    display(Video(out_mp4_path, embed=True, width=640))

In [ ]:
#@title STEP 7 — Batch Scene Processor (JSON Queue + Resume + Auto-Template)
BATCH_JSON_PATH = "/content/longcat_avatar_batch.json" #@param {type:"string"}
START_SCENE = 1 #@param {type:"integer"}
SKIP_EXISTING = True #@param {type:"boolean"}
RESET_RESUME_LOG = False #@param {type:"boolean"}
DEFAULT_RESOLUTION = "480p" #@param ["480p", "720p"]
DEFAULT_ACCELERATION = "balanced" #@param ["balanced", "quality", "fast", "off"]

import json
import os
import re
import time
from pathlib import Path

if "generate_avatar_video" not in globals():
    raise RuntimeError("Please run STEP 3 first to load the LongCat-Video-Avatar-1.5 pipeline.")

batch_file = Path(BATCH_JSON_PATH)
resume_log_path = OUT_DIR / "batch_resume_log.json"

if RESET_RESUME_LOG and resume_log_path.exists():
    resume_log_path.unlink()
    print(f"[Resume Log] Cleared {resume_log_path}")

if not batch_file.exists():
    starter_scenes = [
        {
            "id": "scene_001_single_portrait",
            "mode": "ai2v",
            "image": "/content/LongCat-Video-Avatar-1.5/assets/avatar/examples/photoreal_person.png",
            "audio": "/content/LongCat-Video-Avatar-1.5/assets/avatar/examples/audio/photoreal_person_voice.wav",
            "prompt": "A photorealistic person speaking naturally to the camera in a clean studio portrait, soft neutral background, realistic facial expression.",
            "resolution": "480p",
            "num_frames": 93,
            "num_segments": 1,
            "seed": 42,
        },
        {
            "id": "scene_002_orc_monologue",
            "mode": "ai2v",
            "image": "/content/LongCat-Video-Avatar-1.5/assets/avatar/examples/orc_warrior.png",
            "audio": "/content/LongCat-Video-Avatar-1.5/assets/avatar/examples/audio/orc_warrior_voice.wav",
            "prompt": "A fantasy orc warrior portrait speaking directly to the camera, cinematic studio lighting, detailed armor, expressive face, smoky neutral backdrop.",
            "resolution": "480p",
            "num_frames": 93,
            "num_segments": 1,
            "seed": 77,
        },
        {
            "id": "scene_003_dual_cafe_dialogue",
            "mode": "multi",
            "image": "/content/LongCat-Video-Avatar-1.5/assets/avatar/multi/introduce.png",
            "audio": "/content/LongCat-Video-Avatar-1.5/assets/avatar/multi/introduce_woman.mp3",
            "audio_2": "/content/LongCat-Video-Avatar-1.5/assets/avatar/multi/introduce_man.mp3",
            "audio_type": "add",
            "bbox_1": [100, 80, 800, 640],
            "bbox_2": [50, 720, 820, 1300],
            "prompt": "Static camera, two people are speaking. Two people sit side by side in a warmly lit cafe, engaged in a lively conversation.",
            "resolution": "480p",
            "num_frames": 93,
            "num_segments": 1,
            "seed": 2025,
        },
    ]
    with open(batch_file, "w", encoding="utf-8") as fh:
        json.dump(starter_scenes, fh, indent=2)
    print(f"[Template Created] Wrote 3-scene starter queue to {batch_file}")

with open(batch_file, "r", encoding="utf-8") as fh:
    raw_queue = json.load(fh)

scenes = raw_queue.get("scenes", raw_queue) if isinstance(raw_queue, dict) else raw_queue
if not isinstance(scenes, list):
    raise ValueError(f"Expected a JSON list of scenes in {batch_file}")

resume_log = {}
if resume_log_path.exists():
    try:
        with open(resume_log_path, "r", encoding="utf-8") as fh:
            resume_log = json.load(fh)
    except Exception:
        resume_log = {}

print(f"=== Loaded {len(scenes)} scene(s) from {batch_file} (START_SCENE={START_SCENE}) ===")

for idx, scene in enumerate(scenes, 1):
    scene_id = scene.get("id") or f"batch_{idx:03d}"
    safe_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(scene_id)).strip("_")

    if idx < int(START_SCENE):
        print(f"  [{idx}/{len(scenes)}] SKIP (before START_SCENE {START_SCENE}): {safe_id}")
        continue

    if SKIP_EXISTING:
        logged_mp4 = resume_log.get(safe_id)
        existing_matches = list(OUT_DIR.glob(f"{safe_id}*.mp4"))
        verified_path = None
        if logged_mp4 and Path(logged_mp4).exists() and Path(logged_mp4).stat().st_size > 1024:
            verified_path = Path(logged_mp4)
        elif existing_matches and existing_matches[0].stat().st_size > 1024:
            verified_path = existing_matches[0]
        if verified_path is not None:
            print(f"  [{idx}/{len(scenes)}] SKIP (verified on disk: {verified_path.name})")
            continue

    mode = scene.get("mode", "ai2v")
    image_p = scene.get("image") or scene.get("cond_image")
    audio_p = scene.get("audio")
    audio_p2 = scene.get("audio_2")
    if isinstance(scene.get("cond_audio"), dict):
        audio_p = audio_p or scene["cond_audio"].get("person1")
        audio_p2 = audio_p2 or scene["cond_audio"].get("person2")

    bbox_1 = scene.get("bbox_1")
    bbox_2 = scene.get("bbox_2")
    if isinstance(scene.get("bbox"), dict):
        bbox_1 = bbox_1 or scene["bbox"].get("person1")
        bbox_2 = bbox_2 or scene["bbox"].get("person2")

    prompt = scene.get("prompt", "A person is speaking naturally in front of a steady camera.")
    res = scene.get("resolution", DEFAULT_RESOLUTION)
    n_frames = int(scene.get("num_frames", 93))
    n_segs = int(scene.get("num_segments", 1))
    seed = int(scene.get("seed", 42))
    vocal_m = scene.get("vocal_mode", "separate")
    accel_m = scene.get("accel_mode", DEFAULT_ACCELERATION)
    a_type = scene.get("audio_type", "add")

    print(f"\n  [{idx}/{len(scenes)}] Running '{safe_id}' (mode={mode}, res={res}, frames={n_frames}, segs={n_segs}, seed={seed})")
    try:
        out_mp4 = generate_avatar_video(
            mode=mode,
            image_path=image_p,
            audio_path=audio_p,
            audio_path_2=audio_p2,
            audio_type=a_type,
            bbox_1=bbox_1,
            bbox_2=bbox_2,
            prompt=prompt,
            resolution=res,
            num_frames=n_frames,
            num_segments=n_segs,
            vocal_mode=vocal_m,
            accel_mode=accel_m,
            seed=seed,
            output_stem=safe_id,
        )
        resume_log[safe_id] = str(out_mp4)
        with open(resume_log_path, "w", encoding="utf-8") as fh:
            json.dump(resume_log, fh, indent=2)
    except Exception as exc:
        print(f"  [{idx}/{len(scenes)}] FAIL on '{safe_id}': {exc}")
        torch_gc()